# LangGraph: Agentes con StateGraph y Human-in-the-Loop

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/agentes-avanzados/07-langgraph-agentes.ipynb)

LangGraph permite construir agentes como grafos de estado con flujo explícito, checkpointing y control humano.

In [ ]:
!pip install langgraph langchain-anthropic anthropic -q

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
import operator

# Modelo Claude para LangGraph
llm = ChatAnthropic(model='claude-haiku-4-5-20251001', temperature=0)
print('LangGraph + Claude configurado')

## 1. StateGraph básico

In [ ]:
class EstadoChat(TypedDict):
    mensajes: Annotated[list, operator.add]  # Los mensajes se acumulan
    iteraciones: int
    completado: bool

def nodo_asistente(estado: EstadoChat) -> dict:
    """Nodo que llama a Claude con el historial de mensajes."""
    from langchain_core.messages import HumanMessage
    mensajes = estado['mensajes']
    respuesta = llm.invoke(mensajes)
    return {
        'mensajes': [respuesta],
        'iteraciones': estado['iteraciones'] + 1,
        'completado': True,
    }

def nodo_verificar(estado: EstadoChat) -> str:
    """Decide si continuar o finalizar."""
    if estado['completado'] or estado['iteraciones'] >= 3:
        return 'fin'
    return 'asistente'

# Construir grafo
grafo = StateGraph(EstadoChat)
grafo.add_node('asistente', nodo_asistente)
grafo.set_entry_point('asistente')
grafo.add_conditional_edges('asistente', nodo_verificar, {'fin': END, 'asistente': 'asistente'})

app = grafo.compile()

# Ejecutar
from langchain_core.messages import HumanMessage
resultado = app.invoke({
    'mensajes': [HumanMessage(content='¿Qué es LangGraph en una frase?')],
    'iteraciones': 0,
    'completado': False,
})

print('Respuesta:')
print(resultado['mensajes'][-1].content)

## 2. Checkpointing con MemorySaver

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Añadir persistencia de estado entre llamadas
checkpointer = MemorySaver()
app_con_memoria = grafo.compile(checkpointer=checkpointer)

# Configuración de sesión (thread_id mantiene el estado entre llamadas)
config = {'configurable': {'thread_id': 'sesion-001'}}

# Primera llamada
r1 = app_con_memoria.invoke(
    {'mensajes': [HumanMessage(content='Hola, soy Juan')], 'iteraciones': 0, 'completado': False},
    config=config,
)
print('Turno 1:', r1['mensajes'][-1].content[:100])

# Segunda llamada - recuerda el contexto anterior
r2 = app_con_memoria.invoke(
    {'mensajes': [HumanMessage(content='¿Cómo me llamo?')], 'iteraciones': 0, 'completado': False},
    config=config,
)
print('Turno 2:', r2['mensajes'][-1].content[:100])

## 3. Human-in-the-loop con interrupt

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

class EstadoAprobacion(TypedDict):
    tarea: str
    propuesta: str
    aprobada: bool
    resultado: str

def nodo_planificar(estado: EstadoAprobacion) -> dict:
    response = llm.invoke(f'Propón un plan para: {estado["tarea"]}')
    return {'propuesta': response.content}

def nodo_ejecutar(estado: EstadoAprobacion) -> dict:
    if estado.get('aprobada'):
        response = llm.invoke(f'Ejecuta este plan: {estado["propuesta"]}')
        return {'resultado': response.content}
    return {'resultado': 'Tarea cancelada por el usuario'}

grafo_hitl = StateGraph(EstadoAprobacion)
grafo_hitl.add_node('planificar', nodo_planificar)
grafo_hitl.add_node('ejecutar', nodo_ejecutar)
grafo_hitl.set_entry_point('planificar')
grafo_hitl.add_edge('planificar', 'ejecutar')
grafo_hitl.add_edge('ejecutar', END)

# interrupt_before='ejecutar' pausa antes de ejecutar para aprobación humana
mem = MemorySaver()
app_hitl = grafo_hitl.compile(checkpointer=mem, interrupt_before=['ejecutar'])

config = {'configurable': {'thread_id': 'hitl-001'}}

# Paso 1: planificar (se detiene antes de ejecutar)
estado1 = app_hitl.invoke(
    {'tarea': 'Enviar informe semanal al equipo', 'propuesta': '', 'aprobada': False, 'resultado': ''},
    config=config,
)
print('Plan propuesto:')
print(estado1['propuesta'][:300])

# Paso 2: aprobar y continuar
print('\n--- APROBACIÓN HUMANA ---')
estado_actualizado = {**estado1, 'aprobada': True}
resultado_final = app_hitl.invoke(estado_actualizado, config=config)
print('Resultado:', resultado_final['resultado'][:200])

## Resumen

- **StateGraph**: grafo de nodos que comparten un estado tipado
- **MemorySaver**: checkpointing en memoria entre llamadas
- **interrupt_before/after**: pausa para aprobación humana
- `thread_id` en config mantiene sesiones independientes